In [74]:
import os
import unicodedata

import torch
import pandas as pd
from tqdm import tqdm
import fitz  # PyMuPDF
import concurrent.futures
from functools import partial
import logging
from typing import Dict, List, TypedDict, Any
import unicodedata

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)
from accelerate import Accelerator

# Langchain 관련
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate, ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langgraph.graph import StateGraph, END
from vllm import LLM, SamplingParams


In [59]:
def process_pdf(file_path, chunk_size=800, chunk_overlap=50):
    """PDF 텍스트 추출 후 chunk 단위로 나누기"""
    chunks = []
    try:
        # PDF 파일 열기 (스트림 모드 사용)
        with fitz.open(file_path, filetype="pdf") as doc:
            text = ''
            # 모든 페이지의 텍스트 추출
            for page in doc:
                try:
                    text += page.get_text()
                except Exception as e:
                    logging.warning(f"Error extracting text from page in {file_path}: {str(e)}")
                    continue

            if not text:
                logging.warning(f"No text extracted from {file_path}")
                return chunks

            # 텍스트를 chunk로 분할
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap
            )
            chunk_temp = splitter.split_text(text)
            # Document 객체 리스트 생성
            chunks = [Document(page_content=t) for t in chunk_temp]
    except Exception as e:
        logging.error(f"Error processing {file_path}: {str(e)}")
    
    return chunks

In [60]:
def create_vector_db(chunks, model_path="BAAI/bge-m3"):
    """FAISS DB 생성"""
    if not chunks:
        logging.warning("No chunks provided to create vector DB")
        return None

    try:
        # 임베딩 모델 설정
        model_kwargs = {'device': 'cuda' if torch.cuda.is_available() else 'cpu'}
        encode_kwargs = {'normalize_embeddings': True}
        embeddings = HuggingFaceEmbeddings(
            model_name=model_path,
            model_kwargs=model_kwargs,
            encode_kwargs=encode_kwargs
        )
        # FAISS DB 생성 및 반환
        db = FAISS.from_documents(chunks, embedding=embeddings)
        return db
    except Exception as e:
        logging.error(f"Error creating vector DB: {str(e)}")
        return None


In [61]:
def process_pdfs_from_dataframe(df, base_directory):
    """딕셔너리에 pdf명을 키로해서 DB, retriever 저장"""
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    pdf_databases = {}
    unique_paths = df['Source_path'].unique()
    
    for path in tqdm(unique_paths, desc="Processing PDFs"):
        try:
            # 경로 정규화 및 절대 경로 생성
            normalized_path = normalize_path(path)
            full_path = os.path.normpath(os.path.join(base_directory, normalized_path.lstrip('./'))) if not os.path.isabs(normalized_path) else normalized_path
            
            if not os.path.exists(full_path):
                logging.error(f"File not found: {full_path}")
                continue

            pdf_title = os.path.splitext(os.path.basename(full_path))[0]
            logging.info(f"Processing {pdf_title}...")
            
            # PDF 처리 및 벡터 DB 생성
            chunks = process_pdf(full_path)
            if not chunks:
                logging.warning(f"No chunks extracted from {pdf_title}")
                continue

            db = create_vector_db(chunks)
            if db is None:
                logging.warning(f"Failed to create vector DB for {pdf_title}")
                continue
            
            # Retriever 생성
            retriever = db.as_retriever(search_type="mmr", 
                                        search_kwargs={'k': 3, 'fetch_k': 8})
            
            # 결과 저장
            pdf_databases[pdf_title] = {
                    'db': db,
                    'retriever': retriever
            }
        except Exception as e:
            logging.error(f"Error processing {path}: {str(e)}")
    
    logging.info(f"Processed {len(pdf_databases)} PDFs successfully.")
    return pdf_databases

In [62]:
# Usage
base_directory = './'  # Your Base Directory
df = pd.read_csv('./test.csv')
pdf_databases = process_pdfs_from_dataframe(df, base_directory)

Processing PDFs:   0%|          | 0/9 [00:00<?, ?it/s]2024-08-06 09:32:48,902 - INFO - Processing 중소벤처기업부_혁신창업사업화자금(융자)...
/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
2024-08-06 09:32:51,259 - INFO - PyTorch version 2.3.1 available.
2024-08-06 09:32:51,364 - INFO - Load pretrained SentenceTransformer: BAAI/bge-m3
2024-08-06 09:32:57,016 - INFO - Loading faiss with AVX2 support.
2024-08-06 09:32:57,017 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2024-08-06 09:32:57,018 - INFO - Loadin

In [77]:
def setup_llm_pipeline():
    # 모델 ID 
    model_id = "rtzr/ko-gemma-2-9b-it"

    # 토크나이저 로드 및 설정
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.use_default_system_prompt = False



    # HuggingFacePipeline 객체 생성
    text_generation_pipeline = pipeline(
        model=model_id,
        tokenizer=tokenizer,
        task="text-generation",
        temperature=0.2,
        return_full_text=False,
        max_new_tokens=512,
        device_map="auto"
    )

    llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

    return llm

In [78]:
# LLM 파이프라인
llm = setup_llm_pipeline()

Loading checkpoint shards:   0%|          | 0/10 [00:00<?, ?it/s]

/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 0.3. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFacePipeline`.
  warn_deprecated(


In [79]:
# 상태 정의
class State(TypedDict):
    question: str
    chat_history: List[Dict[str, Any]]
    context: List[Document]
    answer: str

In [80]:
# 노드 함수 정의
def retrieve(state: State) -> State:
    question = state['question']
    relevant_docs = []
    for pdf_info in pdf_databases.values():
        retriever = pdf_info['retriever']
        docs = retriever.get_relevant_documents(question)
        relevant_docs.extend(docs)
    state['context'] = relevant_docs[:3]  # 상위 3개 문서만 사용
    return state

In [83]:
def generate_answer(state: State) -> State:
    question = state['question']
    context = " ".join([doc.page_content for doc in state['context']])
    
    # Hugging Face 모델을 사용하여 답변 생성
    result = llm(question=question, context=context)
    
    state['answer'] = result['answer']
    return state

In [84]:
def should_end(state: State) -> bool:
    return True  # 단순 예제에서는 항상 종료

In [85]:
# 그래프 정의
workflow = StateGraph(State)

# 노드 추가
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate_answer", generate_answer)

# 엣지 추가
workflow.add_edge('retrieve', 'generate_answer')
workflow.add_edge('generate_answer', END)

# 시작 노드 설정
workflow.set_entry_point("retrieve")

# 그래프 컴파일
app = workflow.compile()

In [86]:
# 추론 함수
def infer(question: str) -> str:
    result = app.invoke({"question": question, "chat_history": [], "context": [], "answer": ""})
    return result['answer']

In [87]:
# 사용 예
if __name__ == "__main__":
    question = "What is the main topic of the first PDF?"
    answer = infer(question)
    print(f"Question: {question}")
    print(f"Answer: {answer}")

/home/ubuntu/miniforge3/envs/jinsu/lib/python3.10/site-packages/langchain_core/_api/deprecation.py:139: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


TypeError: BaseLLM.__call__() missing 1 required positional argument: 'prompt'